In [2]:
import xml.etree.ElementTree as ET
import html
import re

class PMC_article:
    """A class defining attributes and methods for parsing a PMC article XML file"""
    
    BLOCK_TAGS = {"p", "sec", "title", "abstract", "body", "list-item", "table-wrap", "fig", "contrib-group", "aff", "list", "disp-formula", "chem-struct-wrap", "tr", "td", "table", "thead", "tbody", "back", "ack", "ref-list", "ref", "app-group", "app", "glossary", "def-list", "def-item", "notes"}
    INLINE_TAGS = {"italic", "bold", "underline", "sup", "sub", "sc", "ext-link", "inline-formula"}
    REF_MAP = {"table": "TABREF", "fig": "FIGREF", "bibr": "BIBREF", "aff": "AFFREF"}
    DOCUMENT_BOUNDARY = "\n\n<|endoftext|>\n"

    def __init__(self, pmc_xml_path):
        self.path = pmc_xml_path
        self._root = None
        self.article_title = None
        self.journal_name = None
        self.abstract_full_text = None
        self.abstract_plain_text = None
        self.abstract_sections = None
        self.body_text = None
        self.body_content = []
        self.body_string=None
        self.abstract_string=None
        self.body_plain_text=None
        self.back_text = None
        self.structured_text = None
        
        self._parse_xml()
        self._extract_metadata()
        self._extract_content()

    def _parse_xml(self):
        """Parses XML file and stores the root Element."""
        try:
            tree = ET.parse(self.path)
            self._root = tree.getroot()
        except Exception as e:
            raise RuntimeError(f"Failed to parse XML file: {self.path}") from e

    def _extract_metadata(self):
        """Extract basic article metadata."""
        title_el = self._root.find(".//article-title")
        if title_el is not None:
            self.article_title = self.get_smart_text(title_el)

        journal_el = self._root.find(".//journal-title")
        if journal_el is not None:
            self.journal_name = self.get_smart_text(journal_el)

    def _extract_content(self):
        """Extract abstract and body content."""
        # Abstract
        abstract_el = self._root.find(".//abstract")
        if abstract_el is not None:
            self.abstract_string = ET.tostring(abstract_el, encoding="unicode", method="xml")
            self.abstract_full_text = self.get_smart_text(abstract_el)
            # Plain text without section titles
            self.abstract_plain_text = self.get_smart_text(abstract_el, ignore_tags={"title"})
            self.abstract_sections = self._parse_sections(abstract_el)

        # Body
        body_el = self._root.find(".//body")
        
        if body_el is not None:
            self.body_string = ET.tostring(body_el, encoding="unicode", method="xml")
            self.body_text = self.get_smart_text(body_el)
            self.body_plain_text = self.get_smart_text(body_el, ignore_tags={"title"})
            
            # Hierarchical extraction for structured data
            self.body_content = []
            for child in body_el:
                if child.tag == "sec":
                    self.body_content.append(self._parse_section_recursive(child))
                elif child.tag == "p" or child.tag in self.BLOCK_TAGS:
                    self.body_content.append({
                        "type": "paragraph",
                        "text": self.get_smart_text(child)
                    })

        # Final structured text for the whole article
        self.structured_text = self._generate_structured_text()

    def _generate_structured_text(self):
        """
        Actually generates the structured text representation.
        """
        parts = []
        # 1. Article Title
        title_el = self._root.find(".//article-title")
        if title_el is not None:
            parts.append(self.get_smart_text(title_el, separator="\n"))

        # 2. Abstract
        abstract_el = self._root.find(".//abstract")
        if abstract_el is not None:
            parts.append("Abstract\n" + self.get_smart_text(abstract_el, separator="\n"))

        # 3. Body
        body_el = self._root.find(".//body")
        if body_el is not None:
            parts.append(self.get_smart_text(body_el, separator="\n"))

        # Join major sections with double newlines
        full_text = "\n\n".join(p for p in parts if p.strip())
        
        return full_text + self.DOCUMENT_BOUNDARY

    def _parse_sections(self, root_el):
        """Flat list of all sections (kept for backward compatibility)."""
        sections = []
        for sec in root_el.findall(".//sec"):
            sections.append(self._parse_section_recursive(sec))
        return sections

    def _parse_section_recursive(self, sec_el):
        """Recursively parse a single section."""
        section = {
            "type": "section",
            "title": None,
            "text": "",
            "subsections": []
        }
        
        # Get title
        title_el = sec_el.find("title")
        if title_el is not None:
            section["title"] = self.get_smart_text(title_el)
            
        # Get direct child paragraphs (exclude those in subsections)
        p_texts = []
        for child in sec_el:
            if child.tag == "p":
                p_texts.append(self.get_smart_text(child))
            elif child.tag == "sec":
                section["subsections"].append(self._parse_section_recursive(child))
            elif child.tag in self.BLOCK_TAGS:
                # Other blocks like lists/tables
                text = self.get_smart_text(child)
                if text:
                    p_texts.append(text)
                
        section["text"] = " ".join(p_texts)
        return section

    @classmethod
    def _extract_equation(cls, formula_el):
        """
        Extract equation content from a formula element.
        Tries to get TeX content first, then MathML, then plain text.
        """
        # Try to find tex-math element
        tex_math = formula_el.find(".//tex-math")
        if tex_math is not None and tex_math.text:
            return tex_math.text.strip()
        
        # Try to find mml:math element (MathML)
        # Note: MathML namespace might be present
        mml_math = formula_el.find(".//{http://www.w3.org/1998/Math/MathML}math")
        if mml_math is None:
            mml_math = formula_el.find(".//math")
        
        if mml_math is not None:
            # For MathML, we'll extract the text content as a fallback
            # A full MathML to LaTeX converter would be complex
            mathml_text = ET.tostring(mml_math, encoding='unicode', method='text').strip()
            if mathml_text:
                return f"[MathML: {mathml_text}]"
        
        # Fallback: extract all text content
        text_content = cls.get_smart_text(formula_el)
        return text_content.strip() if text_content else "equation"

    @staticmethod
    def print_tree_structure(element, level=1):
        """Prints tree structure"""
        for child in element:
            print(" "*level, "-", child.tag)
            self.print_tree_structure(element=child,level=level+1)

    
    @classmethod
    def get_smart_text(cls, element, ignore_tags=None, separator=" "):
        """
        Recursively extract text from an element with smart spacing and reference markers.
        Allows ignoring specific tags (e.g. 'title' to get plain text without headings).
        'separator' controls what is placed around block tags (" " or "\n").
        """
        if element is None:
            return ""

        ignore_tags = ignore_tags or set()
        newline_mode = (separator == "\n")

        def _walk(el, is_root=False):
            tag = el.tag
            
            # Skip entire subtree if tag is ignored
            if tag in ignore_tags:
                return ""

            # Mathematical Equation Formatting
            if tag == "disp-formula":
                # Display equation: wrap in $$...$$
                equation_text = cls._extract_equation(el)
                if newline_mode:
                    return f"\n\n$${equation_text}$$\n\n"
                return f" $${equation_text}$$ "
            
            if tag == "inline-formula":
                # Inline equation: wrap in $...$
                equation_text = cls._extract_equation(el)
                return f"${equation_text}$"

            # Table and Figure Summarization (only if labeled)
            if tag in {"table-wrap", "fig"}:
                label_el = el.find("label")
                if label_el is not None:
                    marker = "TABREF" if tag == "table-wrap" else "FIGREF"
                    label_text = cls.get_smart_text(label_el) if label_el is not None else ""
                    
                    # Try to find a title in the caption
                    title_text = ""
                    caption = el.find("caption")
                    if caption is not None:
                        # Look for title or p inside caption
                        title_el = caption.find(".//title") or caption.find(".//p")
                        if title_el is not None:
                            # Recursively get plain text for the title/caption
                            title_text = cls.get_smart_text(title_el, ignore_tags={"label"})
                    
                    # Clean label: strip leading/trailing brackets if present
                    label_text = label_text.strip("[]").strip()
                    
                    # Ensure title ends with a period for sentence separation
                    title_text = title_text.strip()
                    if title_text:
                        if not title_text.endswith("."):
                            title_text += "."
                        summary = f"{label_text} <{marker}>. {title_text}"
                    else:
                        # If no title, just ensure a period after the marker
                        summary = f"{label_text} <{marker}>."
                    
                    if newline_mode:
                        return f"\n\n{summary}\n\n"
                    return f" {summary} "

            parts = []
            
            # Space/Newline before block elements
            if el.tag in cls.BLOCK_TAGS:
                parts.append(separator)

            # Pre-text (text before first child)
            if el.text:
                parts.append(el.text)
            
            # Recursive walk through children
            for child in el:
                # Styling substitutions (applied before entering child)
                prefix = ""
                if child.tag == "sub":
                    prefix = "_"
                elif child.tag == "sup":
                    prefix = "^"
                
                child_text = _walk(child, is_root=False)
                
                parts.append(prefix + child_text)
                
            # Special handling for xref (Placed AFTER the text/number)
            if tag == "xref":
                ref_type = el.get("ref-type")
                if ref_type in cls.REF_MAP:
                    parts.append(f"<{cls.REF_MAP[ref_type]}>")
            
            # Special handling for ext-link (Append URL)
            if tag == "ext-link":
                url = el.get("{http://www.w3.org/1999/xlink}href") or el.get("href")
                if url:
                    parts.append(f" ({url})")

            # Space/Newline after block elements
            if el.tag in cls.BLOCK_TAGS:
                parts.append(separator)

            # Tail text (text after child closing tag)
            if not is_root and el.tail:
                parts.append(el.tail)
                
            return "".join(parts)

        raw_text = _walk(element, is_root=True)
        return cls.clean_for_llm(raw_text, preserve_newlines=newline_mode)

    @classmethod
    def clean_for_llm(cls, text, preserve_newlines=False):
        """
        Advanced cleaning for LLM training:
        - Decodes entities
        - Normalizes whitespace (optionally preserving newlines)
        - Fixes punctuation spacing
        - Tidies up reference markers
        """
        if not text:
            return ""

        # 1. Decode HTML entities
        text = html.unescape(text)

        # 2. Normalize whitespace
        if preserve_newlines:
            # Collapse horizontal tabs/multiple spaces but keep \n
            text = re.sub(r'[ \t\r]+', ' ', text)
        else:
            # Collapse all whitespace to single space
            text = re.sub(r'\s+', ' ', text)

        # 3. Punctuation spacing cleanup
        text = re.sub(r' +([.,;?!])', r'\1', text)
        # Only add space after punctuation if it's NOT likely a URL or abbreviation
        # Rule: Add space after punctuation if preceded by whitespace, OR if it's lowercase followed by Uppercase.
        text = re.sub(r'(?<= )([.,;?!])([^\s\d\]\)\.,;?!])', r'\1 \2', text)
        text = re.sub(r'([a-z])([.,;?!])([A-Z])', r'\1\2 \3', text)
        
        # 4. Abbreviation protection & joining
        text = re.sub(r'\b([A-Za-z])\.\s+(?=([A-Za-z]\.))', r'\1.', text)
        text = re.sub(r'\bet\s+al\s*\.', 'et al.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bvs\s*\.', 'vs.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bapprox\s*\.', 'approx.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bno\s*\.', 'no.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bvol\s*\.', 'vol.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bref\s*\.', 'ref.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bcf\s*\.', 'cf.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bviz\s*\.', 'viz.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bca\s*\.', 'ca.', text, flags=re.IGNORECASE)
        text = re.sub(r'\bInc\s*\.', 'Inc.', text, flags=re.IGNORECASE)
        
        # 5. Reference marker tidying
        text = re.sub(r'(\b[A-Za-z]+\s+\d+[A-Za-z]?)\s+\d+[A-Za-z]?\b\s*(<[A-Z]+REF>)', r'\1 \2', text)
        text = re.sub(r'(\b\d+[A-Za-z]?)\s+\1\b\s*(<[A-Z]+REF>)', r'\1 \2', text)

        for marker in cls.REF_MAP.values():
            m_tag = f"<{marker}>"
            m = re.escape(m_tag)
            text = re.sub(rf'{m}(?:\s*[,;]\s*{m})+', m_tag, text)
            range_pattern = rf'(\d+)\s*{m}\s*([–\-])\s*(\d+)\s*{m}'
            text = re.sub(range_pattern, rf'\1\2\3 {m_tag}', text)
            text = re.sub(rf'([0-9A-Za-z%]){m}', rf'\1 {m_tag}', text)
            text = re.sub(rf'{m}([0-9A-Za-z])', rf'{m_tag} \1', text)
            text = re.sub(r'([\]\)\}\}])' + m, r'\1 ' + m_tag, text)
            text = re.sub(rf'{m}\s+([.,;:!?])', rf'{m_tag}\1', text)
            
        # 6. Strip unwanted brackets
        text = re.sub(r'\[([\d\s,–\-]*<[A-Z]+REF>.*?)\]', r'\1', text)
            
        # 7. Final whitespace strip
        if preserve_newlines:
            # Collapse multiple spaces
            text = re.sub(r' +', ' ', text)
            # Collapse triple+ newlines to double
            text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
            # Remove spaces at start/end of lines
            text = "\n".join(line.strip() for line in text.split("\n"))
            text = text.strip()
        else:
            text = re.sub(r'\s+', ' ', text).strip()

        return text

    @staticmethod
    def replace_styling_tags(text, replaces={"<sub>": "_", "<sup>": "^"}):
        """Preserve the user's previous styling preference if they still want it."""
        # Note: In the smart walker, we don't have the literal <sub> tags anymore.
        # If we want to keep them, we should handle them in the _walk function.
        return text

# --- Test Section ---
if __name__ == "__main__":
    # Create a dummy XML file for testing
    test_xml = """
    <article>
        <front>
            <journal-title>Journal of Testing</journal-title>
            <article-title>Effect of <sub>X</sub> on <sup>Y</sup></article-title>
            <abstract>
                <sec>
                    <title>Methods</title>
                    <p>We looked at Table 1 <xref ref-type="table">1</xref> and Figure 2 <xref ref-type="fig">2</xref>.</p>
                    <p>Referencing Smith et al. <xref ref-type="bibr">23</xref>.</p>
                </sec>
                <sec>
                    <title>Results</title>
                    <p>Concentration (C<sub>max</sub>) was high.</p>
                </sec>
            </abstract>
        </front>
    </article>
    """
    with open("test_pmc.xml", "w") as f:
        f.write(test_xml)
        
    article = PMC_article("test_pmc.xml")
    print(f"Title: {article.article_title}")
    print(f"Abstract Full: {article.abstract_full_text}")
    print(f"Abstract Plain: {article.abstract_plain_text}")
    for sec in article.abstract_sections:
        print(f"Section [{sec['title']}]: {sec['text']}")


Title: Effect of _X on ^Y
Abstract Full: Methods We looked at Table 1 <TABREF> and Figure 2 <FIGREF>. Referencing Smith et al. 23 <BIBREF>. Results Concentration (C_max) was high.
Abstract Plain: We looked at Table 1 <TABREF> and Figure 2 <FIGREF>. Referencing Smith et al. 23 <BIBREF>. Concentration (C_max) was high.
Section [Methods]: Methods We looked at Table 1 <TABREF> and Figure 2 <FIGREF>. Referencing Smith et al. 23 <BIBREF>.
Section [Results]: Results Concentration (C_max) was high.


In [6]:
article1=PMC_article("./data/entrez_download_PMCID=7067710.xml")

In [5]:
article1.abstract_full_text

'Introduction A fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events. Methods We report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was assessed in healthy adolescents aged 12–17 years, inclusive. Results A total of 35 and 46 subjects were enrolled in the two adult studies, resp

In [4]:
article1.abstract_plain_text

'A fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events. We report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was assessed in healthy adolescents aged 12–17 years, inclusive. A total of 35 and 46 subjects were enrolled in the two adult studies, respectively, and 21 were enrolle

In [5]:
# for element in article1._root.find('.//body').iter():
#     print(element.tag)
def print_tree_structure(element, level=1):
    """Prints tree structure"""
    for child in element:
        print(" "*level, "-", child.tag)
        print_tree_structure(element=child,level=level+1)



print(ET.tostring(article1._root.find('.//body')[3])[0:10000])
# len(ET.tostring(article1._root.find('.//body')[3]))
    

b'<sec xmlns:ns0="http://www.w3.org/1999/xlink" id="Sec7"><title>Results</title><sec id="Sec8"><title>Baseline Characteristics</title><p id="Par22">Baseline characteristics for the three studies are shown in Table&#160;<xref rid="Tab1" ref-type="table">1</xref>. A total of 35 subjects were randomized in Study 1 and 46 in Study 2. In Study 3, 21 subjects were assigned to treatment. In all three studies, the proportion of males and females was approximately 50%. In Study 3, the majority (62%) of subjects were White, whereas in Studies 1 and 2, the largest proportion of subjects were Black (54% and 41%, respectively). One subject in Study 1 discontinued study drug during the first treatment period due to an inability to swallow study medication, and two subjects in Study 2 discontinued study drug during the first treatment period due to difficulties in collecting PK samples; no PK profiling was possible for these two subjects. All 21 subjects in Study 3 completed treatment and were analyz

In [13]:
article1.body_plain_text[3617+11000:1100+3600+12000+12000]

'lity of PK metrics in Studies 1 and 2 was determined by constructing 90% CIs around the estimated difference between test and reference treatments using a mixed-effects model based on natural log-transformed data. The mixed-effects model was implemented using SAS PROC MIXED (SAS Institute, Inc., Cary, NC, USA) with the restricted maximum likelihood estimation method and the Kenward–Roger degrees of freedom algorithm. Because the monocomponent doses in Study 1 were different from those of the FDC, PK metrics were dose normalized to ibuprofen 250 mg and acetaminophen 500 mg for the purposes of comparison. Safety, including AEs, was monitored throughout the in-patient portion of the studies and during a follow-up phone call 14 days after the last investigational drug dose in each study. Baseline characteristics for the three studies are shown in Table 1 <TABREF>. A total of 35 subjects were randomized in Study 1 and 46 in Study 2. In Study 3, 21 subjects were assigned to treatment. In al

In [6]:
print(article1.structured_text)

Phase I Pharmacokinetic Study of Fixed-Dose Combinations of Ibuprofen and Acetaminophen in Healthy Adult and Adolescent Populations

Abstract
Introduction

A fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events.

Methods

We report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was